# Fine-tuned USER-bge-m3: offline evaluation

Notebook отвечает на два вопроса:

1. Улучшает ли Avito LoRA исходный zero-shot USER dense при одинаковом
   chunking и фильтрах?
2. Даёт ли замена dense-канала прирост в BM25+dense RRF и расширяет ли
   candidate pool для следующего supervised LTR?

Global/local weight и RRF parameters выбираются только на прежнем dev-fold.
Test используется для одного primary сравнения. Это пятый model-family look,
поэтому Bonferroni threshold равен `0.01`, а bootstrap confidence — 99%.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

import pandas as pd
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.metrics import recall_at_k
from avito_retrieval.statistics import paired_recall_test

FINE_DIR = ROOT / "artifacts/finetuned_dense_kaggle"
PRIMARY_ALPHA = 0.01
PRIMARY_CONFIDENCE = 0.99


def parse_rank(value: object) -> list[str]:
    return [] if value is None or pd.isna(value) or not str(value) else str(value).split()


def subset(mapping: dict, ids: set[str]) -> dict:
    return {query_id: mapping[query_id] for query_id in ids}


def fuse_row(row: pd.Series, global_column: str, local_column: str, weight: float, top_k: int = 250) -> list[str]:
    return reciprocal_rank_fusion(
        [parse_rank(row[global_column]), parse_rank(row[local_column])],
        weights=[1.0, weight], rrf_k=60, top_k=top_k,
    )

## 1. Artifact contract и неизменный split

Проверяются query coverage, уникальность items, принадлежность corpus и полные
top-250 global rankings. Split пересоздаётся тем же seed/stratum, а не
выбирается после просмотра fine-tuned результатов.

In [ ]:
manifest = pd.read_parquet(ROOT / "artifacts/validation/manifest.parquet").copy()
labels = pd.read_parquet(ROOT / "artifacts/validation/labels.parquet")
benchmark = pd.read_parquet(ROOT / "dataset/benchmark_queries.parquet")
items = pd.read_parquet(ROOT / "dataset/benchmark_items.parquet", columns=["item_id"])
valid_items = set(items.item_id.astype(str))
relevant = labels.groupby("eval_query_id").item_id.agg(set).to_dict()

fine = pd.read_parquet(FINE_DIR / "finetuned_dense_rankings.parquet")
run = json.loads((FINE_DIR / "finetuned_dense_run.json").read_text())
zero = pd.read_parquet(ROOT / "artifacts/dense_kaggle/dense_rankings.parquet")
bm25 = pd.read_parquet(ROOT / "artifacts/rankings/bm25_rankings.parquet")
splade = pd.read_parquet(ROOT / "artifacts/splade_kaggle/splade_rankings.parquet")

assert len(fine) == len(manifest) + len(benchmark) and fine.query_key.is_unique
assert set(fine[fine.split.eq("validation")].query_key) == set(manifest.eval_query_id)
assert set(fine[fine.split.eq("benchmark")].query_key) == set(benchmark.query_id)
for column in ["finetuned_global", "finetuned_local"]:
    for ranking in fine[column].map(parse_rank):
        assert len(ranking) <= 250 and len(ranking) == len(set(ranking))
        assert set(ranking) <= valid_items
assert fine.finetuned_global.map(parse_rank).map(len).eq(250).all()

manifest["fold"] = -1
splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
for fold, (_, indices) in enumerate(splitter.split(manifest, manifest.primary_stratum)):
    manifest.loc[indices, "fold"] = fold
dev_ids = set(manifest.loc[manifest.fold.eq(0), "eval_query_id"])
test_ids = set(manifest.loc[manifest.fold.eq(1), "eval_query_id"])
display(run)

## 2. Local-channel ablation на dev

Проверяется маленькая заранее заданная сетка. Значение выбирается по dev
Recall@50; test-метрики на этом шаге не вычисляются.

In [ ]:
joined = (
    fine.merge(zero, on=["query_key", "split"], validate="one_to_one")
    .merge(bm25, on=["query_key", "split"], validate="one_to_one")
    .merge(splade, on=["query_key", "split"], validate="one_to_one")
)
validation = joined[joined.split.eq("validation")].set_index("query_key")
local_grid, fine_cache = [], {}
for weight in [0.75, 1.0, 1.15, 1.25, 1.5, 2.0, 3.0]:
    prediction = {
        query_id: fuse_row(row, "finetuned_global", "finetuned_local", weight)
        for query_id, row in validation.iterrows()
    }
    fine_cache[weight] = prediction
    local_grid.append({
        "local_weight": weight,
        "dev_recall": recall_at_k(subset(prediction, dev_ids), subset(relevant, dev_ids)),
    })
local_grid = pd.DataFrame(local_grid).sort_values(
    ["dev_recall", "local_weight"], ascending=[False, True]
)
selected_local_weight = float(local_grid.iloc[0].local_weight)
fine_pred = fine_cache[selected_local_weight]
zero_pred = {
    query_id: fuse_row(row, "dense_global", "dense_local", 1.15)
    for query_id, row in validation.iterrows()
}
display(local_grid)
print("selected local weight on dev:", selected_local_weight)

## 3. BM25+dense replacement

Для fine-tuned канала `rrf_k` и dense weight выбираются только на dev. Control
— ранее зафиксированный BM25+zero-shot dense (`k=20`, dense weight 1.25).
Primary endpoint сравнивает эти два полных candidate generators, а не только
isolated dense модели.

In [ ]:
bm25_pred = {query_id: parse_rank(row.bm25) for query_id, row in validation.iterrows()}
control_hybrid = {
    query_id: reciprocal_rank_fusion(
        [bm25_pred[query_id], zero_pred[query_id]],
        weights=[1.0, 1.25], rrf_k=20, top_k=50,
    )
    for query_id in relevant
}
grid, hybrid_cache = [], {}
for rrf_k in [10, 20, 40, 60, 100]:
    for dense_weight in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
        prediction = {
            query_id: reciprocal_rank_fusion(
                [bm25_pred[query_id], fine_pred[query_id]],
                weights=[1.0, dense_weight], rrf_k=rrf_k, top_k=50,
            )
            for query_id in relevant
        }
        hybrid_cache[(rrf_k, dense_weight)] = prediction
        grid.append({
            "rrf_k": rrf_k, "dense_weight": dense_weight,
            "dev_recall": recall_at_k(subset(prediction, dev_ids), subset(relevant, dev_ids)),
        })
grid = pd.DataFrame(grid).sort_values(
    ["dev_recall", "dense_weight"], ascending=[False, True]
)
best = grid.iloc[0]
best_key = (int(best.rrf_k), float(best.dense_weight))
fine_hybrid = hybrid_cache[best_key]
display(grid.head(12))

primary = asdict(paired_recall_test(
    fine_hybrid, control_hybrid, relevant, query_ids=sorted(test_ids),
    confidence=PRIMARY_CONFIDENCE,
))
dense_secondary = asdict(paired_recall_test(
    fine_pred, zero_pred, relevant, query_ids=sorted(test_ids),
))
comparison = pd.DataFrame([
    {"method": "zero-shot dense", "dev": recall_at_k(subset(zero_pred, dev_ids), subset(relevant, dev_ids)),
     "test": recall_at_k(subset(zero_pred, test_ids), subset(relevant, test_ids))},
    {"method": "fine-tuned dense", "dev": recall_at_k(subset(fine_pred, dev_ids), subset(relevant, dev_ids)),
     "test": recall_at_k(subset(fine_pred, test_ids), subset(relevant, test_ids))},
    {"method": "BM25+zero", "dev": recall_at_k(subset(control_hybrid, dev_ids), subset(relevant, dev_ids)),
     "test": recall_at_k(subset(control_hybrid, test_ids), subset(relevant, test_ids))},
    {"method": "BM25+fine", "dev": recall_at_k(subset(fine_hybrid, dev_ids), subset(relevant, dev_ids)),
     "test": recall_at_k(subset(fine_hybrid, test_ids), subset(relevant, test_ids))},
])
display(comparison)
display(pd.DataFrame({"primary_BM25_fine_vs_zero": primary, "dense_only": dense_secondary}).T)
channel_accepted = bool(
    primary["mean_delta"] > 0 and primary["ci_low"] > 0
    and primary["p_value_greater"] < PRIMARY_ALPHA
)
print("fine-tuned channel accepted:", channel_accepted)

## 4. Candidate-pool ceiling для следующего LTR

Union — диагностический широкий пул, не допустимый answer@50. Если fine-tuned
dense статистически принят и расширяет pool, следующий notebook добавляет его
ranks/features в CatBoost. Иначе дорогой канал не усложняет pipeline.

In [ ]:
old_union, new_union = {}, {}
for query_id, row in validation.iterrows():
    splade_rank = reciprocal_rank_fusion(
        [parse_rank(row.splade_q32_d192_global), parse_rank(row.splade_q32_d192_local)],
        weights=[1.0, 2.0], rrf_k=60, top_k=200,
    )
    old = list(dict.fromkeys([
        *bm25_pred[query_id][:200], *zero_pred[query_id][:200], *splade_rank,
    ]))
    new = list(dict.fromkeys([*old, *fine_pred[query_id][:200]]))
    old_union[query_id], new_union[query_id] = old, new
pool_test = {
    "old_union": recall_at_k(subset(old_union, test_ids), subset(relevant, test_ids), k=600),
    "with_finetuned_dense": recall_at_k(subset(new_union, test_ids), subset(relevant, test_ids), k=800),
}
pool_paired = asdict(paired_recall_test(
    new_union, old_union, relevant, query_ids=sorted(test_ids), k=800,
))
print(pool_test)
display(pd.DataFrame({"new_pool_vs_old": pool_paired}).T)

metrics = {
    "run": run,
    "selected_local_weight": selected_local_weight,
    "selected_rrf_k": best_key[0],
    "selected_dense_weight": best_key[1],
    "comparison": comparison.to_dict("records"),
    "primary_alpha": PRIMARY_ALPHA,
    "bootstrap_confidence": PRIMARY_CONFIDENCE,
    "primary_test": primary,
    "dense_only_test": dense_secondary,
    "channel_accepted": channel_accepted,
    "candidate_pool": pool_test,
    "candidate_pool_test": pool_paired,
}
(ROOT / "reports/finetuned_dense_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)